# 02 — MLP with Flax + Optax

**Optional Track · Lê Nguyễn Ngọc Vũ**

This notebook covers:
- Defining a neural network with Flax `linen`
- Initializing parameters (explicit pytree, not hidden state)
- Optimizing with Optax (Adam)
- Full training loop with `jit`
- Side-by-side comparison with the PyTorch equivalent

**Same task as `module_1_nn_cnn/notebooks/02_mlp_regression.ipynb`** — heat flux prediction — so you can directly compare code style and performance.

---

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

print(f'JAX: {jax.__version__}')
import flax; print(f'Flax: {flax.__version__}')
print(f'Optax: {optax.__version__}')

key = jax.random.PRNGKey(42)

## 1. Generate Dataset (Same as PyTorch Notebook)

In [ ]:
# Heat flux: Q = h * A * delta_T
N = 2000
key, k1, k2, k3, kn = jax.random.split(key, 5)

h       = jax.random.uniform(k1, (N, 1)) * 100 + 10
A       = jax.random.uniform(k2, (N, 1)) * 0.5 + 0.01
delta_T = jax.random.uniform(k3, (N, 1)) * 80 + 5

X = jnp.concatenate([h, A, delta_T], axis=1)   # (N, 3)
y = (h * A * delta_T)[:, 0]                     # (N,)  — scalar output
y = y + 0.02 * y * jax.random.normal(kn, (N,))  # add noise

# Normalize
X_mean, X_std = X.mean(0), X.std(0)
y_mean, y_std = y.mean(), y.std()
X_norm = (X - X_mean) / X_std
y_norm = (y - y_mean) / y_std

# Train / val split
split = int(0.8 * N)
X_train, X_val = X_norm[:split], X_norm[split:]
y_train, y_val = y_norm[:split], y_norm[split:]

print(f'Train: {X_train.shape}, Val: {X_val.shape}')

## 2. Define MLP with Flax

In [ ]:
class MLP(nn.Module):
    hidden_dims: tuple = (128, 128, 64)
    out_dim: int = 1

    @nn.compact
    def __call__(self, x):
        for h in self.hidden_dims:
            x = nn.Dense(h)(x)
            x = nn.relu(x)
        return nn.Dense(self.out_dim)(x)[:, 0]  # (batch,)

model = MLP()

# Initialize — JAX requires explicit shapes
key, init_key = jax.random.split(key)
params = model.init(init_key, jnp.ones((1, 3)))['params']

# Inspect structure
print('Parameter tree structure:')
for layer, p in params.items():
    for name, val in p.items():
        print(f'  {layer}/{name}: {val.shape}')

## 3. Loss Function and Training Step

In [ ]:
def mse_loss(params, X, y):
    preds = model.apply({'params': params}, X)
    return jnp.mean((preds - y) ** 2)

# Optimizer
optimizer = optax.adam(learning_rate=1e-3)
opt_state = optimizer.init(params)

@jax.jit
def train_step(params, opt_state, X_batch, y_batch):
    loss, grads = jax.value_and_grad(mse_loss)(params, X_batch, y_batch)
    updates, new_opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state, loss

@jax.jit
def eval_loss(params, X, y):
    return mse_loss(params, X, y)

print('Train step compiled (first call will be slow — that is expected).')

## 4. Training Loop

In [ ]:
BATCH_SIZE = 64
NUM_EPOCHS = 100
N_TRAIN = X_train.shape[0]

train_losses, val_losses = [], []

for epoch in range(NUM_EPOCHS):
    # Shuffle training data
    key, shuffle_key = jax.random.split(key)
    perm = jax.random.permutation(shuffle_key, N_TRAIN)
    X_shuffled, y_shuffled = X_train[perm], y_train[perm]

    epoch_loss = 0.0
    n_batches = N_TRAIN // BATCH_SIZE

    for i in range(n_batches):
        X_batch = X_shuffled[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        y_batch = y_shuffled[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
        params, opt_state, loss = train_step(params, opt_state, X_batch, y_batch)
        epoch_loss += loss

    train_losses.append(float(epoch_loss / n_batches))
    val_losses.append(float(eval_loss(params, X_val, y_val)))

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS}  '
              f'Train: {train_losses[-1]:.4f}  Val: {val_losses[-1]:.4f}')

## 5. Plot Loss Curves

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train loss')
plt.plot(val_losses, label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('MSE (normalized)')
plt.title('Flax MLP — Heat Flux Regression')
plt.legend()
plt.grid(True)
plt.show()

## 6. Evaluate: Predicted vs True

In [ ]:
y_pred_norm = model.apply({'params': params}, X_val)
y_pred = np.array(y_pred_norm) * float(y_std) + float(y_mean)
y_true = np.array(y_val) * float(y_std) + float(y_mean)

mse = np.mean((y_pred - y_true) ** 2)
mae = np.mean(np.abs(y_pred - y_true))
print(f'Val MSE (original scale): {mse:.2f}')
print(f'Val MAE (original scale): {mae:.2f}')

plt.figure(figsize=(5, 5))
plt.scatter(y_true, y_pred, alpha=0.3, s=10)
lim = [y_true.min(), y_true.max()]
plt.plot(lim, lim, 'r--', label='y = x')
plt.xlabel('True Q')
plt.ylabel('Predicted Q')
plt.title('Predicted vs True (Flax MLP)')
plt.legend()
plt.grid(True)
plt.show()

## 7. PyTorch vs JAX/Flax — Side-by-Side Comparison

| | PyTorch | JAX / Flax |
|--|---------|------------|
| Model definition | `class Net(nn.Module): def forward(self, x)` | `class Net(nn.Module): def __call__(self, x)` |
| Parameters | Hidden in model object | Explicit pytree dict |
| Forward pass | `model(x)` | `model.apply({'params': params}, x)` |
| Gradient | `loss.backward()` | `jax.grad(loss_fn)(params, ...)` |
| Optimizer step | `optim.step()` | `optax.update + optax.apply_updates` |
| JIT | `torch.compile(model)` | `@jax.jit` on train_step function |
| State | Mutable in model | Carried explicitly |

**Key insight:** In JAX, there is no hidden state. Parameters, optimizer state, and RNG keys are all **values you pass around explicitly**. This makes distributed training and checkpointing much simpler.
